# 🚗 KSI Toronto — Step 4: Prediction

## Objective

The objective of this notebook is to load the saved best machine learning model and use it to make predictions on collision cases.

In this step, we will:
- load the final Random Forest model
- load the prepared test dataset
- generate predictions
- compare predicted results with actual values
- create a simple prediction workflow for future deployment

In [1]:
# ============================================================
# Step 4 — Prediction
# ============================================================

import pickle
import pandas as pd
import numpy as np

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 4.1 Load Final Model and Test Data

In this section, we load the saved Random Forest model and the prepared test dataset.  
This allows us to generate predictions using the best model selected in Step 3.

In [2]:
# ============================================================
# 4.1 — Load Final Model and Test Data
# ============================================================

MODELS_PATH = r'C:\Users\HP\Desktop\Artificial Intelligence - Software Engineering Technology (Fast-Track)\26W --Supervised Learning (SEC. 402)\COMP247_KSI_Project\models\\'

# Load final model
with open(MODELS_PATH + 'best_random_forest_model.pkl', 'rb') as file:
    final_model = pickle.load(file)

# Load test data
X_test = pickle.load(open(MODELS_PATH + 'X_test.pkl', 'rb'))
y_test = pickle.load(open(MODELS_PATH + 'y_test.pkl', 'rb'))

print("✅ Final model and test data loaded successfully")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

✅ Final model and test data loaded successfully
X_test shape: (3788, 34)
y_test shape: (3788,)


## 4.2 Generate Predictions

In this section, the saved Random Forest model is used to generate predictions on the test dataset.

The model predicts whether each collision case is:
- `0` = Non-Fatal
- `1` = Fatal

In [3]:
# ============================================================
# 4.2 — Generate Predictions
# ============================================================

# Generate class predictions
y_pred = final_model.predict(X_test)

# Generate prediction probabilities for the Fatal class
y_prob = final_model.predict_proba(X_test)[:, 1]

print("✅ Predictions generated successfully")
print(f"Total predictions: {len(y_pred):,}")
print(f"Predicted Non-Fatal cases: {(y_pred == 0).sum():,}")
print(f"Predicted Fatal cases    : {(y_pred == 1).sum():,}")

✅ Predictions generated successfully
Total predictions: 3,788
Predicted Non-Fatal cases: 3,502
Predicted Fatal cases    : 286


## 4.3 Compare Actual and Predicted Results

In this section, we compare the model predictions with the actual test labels.  
This helps verify how the saved model performs on unseen data.

In [4]:
# ============================================================
# 4.3 — Compare Actual and Predicted Results
# ============================================================

prediction_results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred,
    "Fatal_Probability": y_prob
})

# Add readable labels
prediction_results["Actual_Label"] = prediction_results["Actual"].map({
    0: "Non-Fatal",
    1: "Fatal"
})

prediction_results["Predicted_Label"] = prediction_results["Predicted"].map({
    0: "Non-Fatal",
    1: "Fatal"
})

# Reorder columns
prediction_results = prediction_results[
    ["Actual", "Actual_Label", "Predicted", "Predicted_Label", "Fatal_Probability"]
]

print("✅ Prediction results table created successfully")
prediction_results.head(10)

✅ Prediction results table created successfully


,Actual,Actual_Label,Predicted,Predicted_Label,Fatal_Probability
13625,1,Fatal,1,Fatal,0.94
10570,0,Non-Fatal,0,Non-Fatal,0.12
12908,0,Non-Fatal,0,Non-Fatal,0.02
11089,0,Non-Fatal,0,Non-Fatal,0.00
7523,0,Non-Fatal,0,Non-Fatal,0.19
2089,0,Non-Fatal,0,Non-Fatal,0.01
18097,0,Non-Fatal,0,Non-Fatal,0.30
12963,0,Non-Fatal,0,Non-Fatal,0.13
18106,0,Non-Fatal,0,Non-Fatal,0.18
17677,0,Non-Fatal,0,Non-Fatal,0.03


In [5]:
# ============================================================
# 4.4 — Clean Prediction Results Display
# ============================================================

prediction_results_clean = prediction_results.reset_index(drop=True)

print("✅ Clean prediction table preview")
prediction_results_clean.head(10)

✅ Clean prediction table preview


,Actual,Actual_Label,Predicted,Predicted_Label,Fatal_Probability
0,1,Fatal,1,Fatal,0.94
1,0,Non-Fatal,0,Non-Fatal,0.12
2,0,Non-Fatal,0,Non-Fatal,0.02
3,0,Non-Fatal,0,Non-Fatal,0.00
4,0,Non-Fatal,0,Non-Fatal,0.19
5,0,Non-Fatal,0,Non-Fatal,0.01
6,0,Non-Fatal,0,Non-Fatal,0.30
7,0,Non-Fatal,0,Non-Fatal,0.13
8,0,Non-Fatal,0,Non-Fatal,0.18
9,0,Non-Fatal,0,Non-Fatal,0.03


## 4.5 Prediction Correctness Analysis

In this section, we check whether each prediction matches the actual label.  
This helps us understand how many predictions were correct and how many were incorrect.

In [6]:
# ============================================================
# 4.5 — Prediction Correctness Analysis
# ============================================================

prediction_results_clean["Correct_Prediction"] = (
    prediction_results_clean["Actual"] == prediction_results_clean["Predicted"]
)

correct_count = prediction_results_clean["Correct_Prediction"].sum()
incorrect_count = len(prediction_results_clean) - correct_count

print("✅ Prediction correctness calculated successfully")
print(f"Correct predictions   : {correct_count:,}")
print(f"Incorrect predictions : {incorrect_count:,}")
print(f"Total predictions     : {len(prediction_results_clean):,}")

prediction_results_clean.head(10)

✅ Prediction correctness calculated successfully
Correct predictions   : 3,528
Incorrect predictions : 260
Total predictions     : 3,788


,Actual,Actual_Label,Predicted,Predicted_Label,Fatal_Probability,Correct_Prediction
0,1,Fatal,1,Fatal,0.94,True
1,0,Non-Fatal,0,Non-Fatal,0.12,True
2,0,Non-Fatal,0,Non-Fatal,0.02,True
3,0,Non-Fatal,0,Non-Fatal,0.00,True
4,0,Non-Fatal,0,Non-Fatal,0.19,True
5,0,Non-Fatal,0,Non-Fatal,0.01,True
6,0,Non-Fatal,0,Non-Fatal,0.30,True
7,0,Non-Fatal,0,Non-Fatal,0.13,True
8,0,Non-Fatal,0,Non-Fatal,0.18,True
9,0,Non-Fatal,0,Non-Fatal,0.03,True


## 4.6 Analyze Incorrect Predictions

In this section, we display examples where the model prediction did not match the actual label.  
This helps us better understand the model limitations.

In [7]:
# ============================================================
# 4.6 — Analyze Incorrect Predictions
# ============================================================

incorrect_predictions = prediction_results_clean[
    prediction_results_clean["Correct_Prediction"] == False
]

print("✅ Incorrect predictions extracted successfully")
print(f"Total incorrect predictions: {len(incorrect_predictions):,}")

incorrect_predictions.head(10)

✅ Incorrect predictions extracted successfully
Total incorrect predictions: 260


,Actual,Actual_Label,Predicted,Predicted_Label,Fatal_Probability,Correct_Prediction
16,1,Fatal,0,Non-Fatal,0.24,False
27,1,Fatal,0,Non-Fatal,0.39,False
40,1,Fatal,0,Non-Fatal,0.28,False
55,1,Fatal,0,Non-Fatal,0.33,False
57,1,Fatal,0,Non-Fatal,0.35,False
92,1,Fatal,0,Non-Fatal,0.31,False
116,1,Fatal,0,Non-Fatal,0.26,False
124,1,Fatal,0,Non-Fatal,0.30,False
130,1,Fatal,0,Non-Fatal,0.08,False
132,1,Fatal,0,Non-Fatal,0.25,False


### Error Analysis Interpretation

Most incorrect predictions shown above are fatal collisions that were predicted as non-fatal.  
This means that although the Random Forest model performs very well overall, it still misses some fatal collision cases.

This limitation is important because fatal collisions are the minority class and are harder for the model to detect.  
Future improvements could focus on increasing recall for fatal collisions by adjusting the prediction threshold, tuning model parameters, or testing additional imbalance-handling techniques.

## 4.7 Save Prediction Results

In this section, the prediction results are saved as a CSV file.  
This file can be used for reporting, model analysis, or future deployment testing.

In [8]:
# ============================================================
# 4.7 — Save Prediction Results
# ============================================================

PREDICTION_PATH = r'C:\Users\HP\Desktop\Artificial Intelligence - Software Engineering Technology (Fast-Track)\26W --Supervised Learning (SEC. 402)\COMP247_KSI_Project\models\\prediction_results.csv'

prediction_results_clean.to_csv(PREDICTION_PATH, index=False)

print("✅ Prediction results saved successfully")
print(f"File saved at: {PREDICTION_PATH}")
print(f"Rows saved: {len(prediction_results_clean):,}")

✅ Prediction results saved successfully
File saved at: C:\Users\HP\Desktop\Artificial Intelligence - Software Engineering Technology (Fast-Track)\26W --Supervised Learning (SEC. 402)\COMP247_KSI_Project\models\\prediction_results.csv
Rows saved: 3,788
